<a href="https://colab.research.google.com/github/lipikamallikarjuna/MissingRfidDetection-ANSC6060/blob/main/MissingRFIDPipelineForReproducibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import and Paths

In [1]:

# CELL 1 Imports & Paths

from pathlib import Path
import numpy as np
import pandas as pd
import pickle
from scipy.optimize import curve_fit, linear_sum_assignment

from google.colab import drive
drive.mount("/content/drive")

# Paths
ProjectRoot   = Path("/content/drive/MyDrive/MiniProject")
DataFilePath  = ProjectRoot / "Data_set_prep_assignment_1 .csv"

# Make sure utils module is importable
import sys
sys.path.insert(0, str(ProjectRoot))
from pipeline_utils import (
    load_and_clean, build_gaps, build_animals_by_date,
    build_session_arrays, fit_woods_params, predict_woods,
    precompute_woods_predictions, save_artifacts, load_artifacts,
    data_quality_report, woods_curve
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# CELL 2 Load & Clean

DfKnown, DfMissing, DfMaster = load_and_clean(DataFilePath)

[load_and_clean] DfKnown   : 6,794,418 rows
[load_and_clean] DfMissing : 1,701,003 rows
[load_and_clean] DfMaster  : 8,495,421 rows


In [4]:
# CELL 3 Data Quality Report

report = data_quality_report(DfMaster)
display(report)

,dtype,missing_pct,unique_vals,neg_count,zero_count
AnimalId,string,20.02,9087,0,0
LactationNumber,float64,20.02,12,0,0
DaysInMilk,float64,20.02,870,0,0
ReproductionStatus,object,20.02,4,0,0
EventDate,datetime64[ns],0.00,830,0,0
AverageMilkFlowKgPerMin,float64,0.00,219,0,215
MilkFlow30To60SecondsKgPerMin,float64,0.00,102,0,3155
MilkYieldFirst2MinutesKg,float64,0.00,1675,0,0
TotalMilkYieldSessionKg,float64,0.00,1010,0,0
MilkingDurationSeconds,int64,0.00,604,0,0


In [5]:
# CELL 4 Confirm Split

print(f"Known   : {len(DfKnown):,}  rows — have AnimalId")
print(f"Missing : {len(DfMissing):,}  rows — mystery records")
print(f"Total   : {len(DfMaster):,}  rows")

# Quick missingness sanity check
id_cols = ["AnimalId", "LactationNumber", "DaysInMilk", "ReproductionStatus"]
pattern = DfMaster[id_cols].isna().astype(int).value_counts()
print("\nMissingness pattern (should be only 0000 and 1111):")
print(pattern)

Known   : 6,794,418  rows — have AnimalId
Missing : 1,701,003  rows — mystery records
Total   : 8,495,421  rows

Missingness pattern (should be only 0000 and 1111):
AnimalId  LactationNumber  DaysInMilk  ReproductionStatus
0         0                0           0                     6794414
1         1                1           1                     1701003
0         0                1           0                           4
Name: count, dtype: int64


In [6]:
# CELL 5 Gap Detection

#DfGaps = build_gaps(DfKnown, min_gap_days=30)
#print(DfGaps.head(3))

# Instead of re-run — just load the saved gaps
DfGaps = pd.read_csv(ProjectRoot / "DfGaps.csv", parse_dates=["GapStart","GapEnd","PreDate","PostDate","CalvingDate"])
print(f"Loaded {len(DfGaps):,} gap periods")
print(DfGaps.head(3))

Loaded 8,930 gap periods
              AnimalId    PreDate   GapStart     GapEnd  GapDays  PreLac  \
0 -1007048835775197250 2020-01-21 2020-01-22 2020-03-27       66     1.0   
1 -1007048835775197250 2021-03-11 2021-03-12 2021-07-08      119     2.0   
2 -1007468268464577766 2019-07-25 2019-07-26 2019-09-07       44     2.0   

   PreDIM  PreRepro   PreYield   PreFlow  PreDuration   PostDate  PostLac  \
0   289.0  Pregnant  12.383072  4.581283          160 2020-03-28      2.0   
1   349.0  Pregnant   5.579186  2.494758          133 2021-07-09      3.0   
2   219.0  Pregnant  14.061363  3.991613          207 2019-09-08      2.0   

   PostDIM PostRepro  PostYield CalvingDate  
0      2.0     Fresh  15.331422  2020-03-26  
1      1.0     Fresh  13.290256  2021-07-08  
2    264.0  Pregnant   9.570799         NaT  


In [7]:

# CELL 6 Build Lookups

AnimalsByDate  = build_animals_by_date(DfGaps)
SessionArrays  = build_session_arrays(DfMissing)

[build_animals_by_date] Dates covered     : 861
[build_animals_by_date] Animal-day entries: 598,672
[build_session_arrays] Unique (date, session) slots: 2,490


In [ ]:

# CELL 7 Fit Wood's Curves

AllWoodsParams = fit_woods_params(DfKnown)

In [ ]:
# CELL 8 Precompute Wood's Predictions

WoodsPredDictAll = precompute_woods_predictions(AllWoodsParams, DfGaps)

In [ ]:
# CELL 9 Hungarian Matching

# Pre-compute yield statistics for cost normalisation
YieldStd  = DfKnown["TotalMilkYieldSessionKg"].std()
FlowStd   = DfKnown["AverageMilkFlowKgPerMin"].std()
DurStd    = DfKnown["MilkingDurationSeconds"].std()

Results = []   # will hold all assignment rows

for Date, AbsentAnimals in AnimalsByDate.items():
    for Session in [1, 2, 3]:
        MysteryIdx = SessionArrays.get((Date, Session))
        if MysteryIdx is None or len(MysteryIdx) == 0:
            continue

        MysteryRecs = DfMissing.loc[MysteryIdx]
        RecYields   = MysteryRecs["TotalMilkYieldSessionKg"].values
        RecFlows    = MysteryRecs["AverageMilkFlowKgPerMin"].values
        RecDurs     = MysteryRecs["MilkingDurationSeconds"].values

        # Filter to animals with a Wood's prediction for this date
        ValidAnimals = [a for a in AbsentAnimals
                        if (a, Date) in WoodsPredDictAll]
        if not ValidAnimals:
            continue

        n_animals = len(ValidAnimals)
        n_records = len(MysteryIdx)
        n         = max(n_animals, n_records)
        CostMatrix = np.full((n, n), fill_value=1e6)

        for j, aid in enumerate(ValidAnimals):
            ExpYield  = WoodsPredDictAll[(aid, Date)]
            YieldDiff = np.abs(RecYields - ExpYield)

            # Pre-filter: skip if yield difference > 2 std
            valid_mask = YieldDiff <= 2 * YieldStd
            if not valid_mask.any():
                continue

            CostMatrix[j, :n_records] = (
                10 * YieldDiff / YieldStd +
                np.abs(RecFlows - RecFlows.mean()) / (FlowStd + 1e-9) +
                np.abs(RecDurs  - RecDurs.mean())  / (DurStd  + 1e-9)
            )
            CostMatrix[j, ~valid_mask[:n_records]] = 1e6   # block invalid

        RowInd, ColInd = linear_sum_assignment(CostMatrix)

        for r, c in zip(RowInd, ColInd):
            if r >= n_animals or c >= n_records:
                continue
            if CostMatrix[r, c] >= 1e6:
                continue
            aid  = ValidAnimals[r]
            ridx = MysteryIdx[c]
            Results.append({
                "OriginalIndex":       ridx,
                "RecoveredAnimalId":   aid,
                "Date":                Date,
                "Session":             Session,
                "AssignedCost":        CostMatrix[r, c],
                "ExpectedYield":       WoodsPredDictAll[(aid, Date)],
                "ActualYield":         RecYields[c],
            })

DfHungarian = pd.DataFrame(Results)
print(f"[Hungarian] Total assignments : {len(DfHungarian):,}")

In [ ]:
# CELL 10 Confidence Filter

CONFIDENCE_THRESHOLD_KG = 0.5   # |actual - predicted| < 0.5 kg

DfHungarian["YieldError"] = np.abs(
    DfHungarian["ActualYield"] - DfHungarian["ExpectedYield"]
)
DfHighConf = DfHungarian[DfHungarian["YieldError"] < CONFIDENCE_THRESHOLD_KG].copy()

pct = len(DfHighConf) / len(DfHungarian) * 100
print(f"High-confidence assignments : {len(DfHighConf):,}  ({pct:.1f}%)")

In [ ]:
# VALIDATION CELL Simulated tag-loss test on known animals
import random

# Pick 200 random known animals to "hide" and test against
random.seed(42)
TestAnimals = random.sample(list(DfKnown["AnimalId"].unique()), 200)

# Remove them from known pool — pretend their tags failed
DfKnownHeld  = DfKnown[~DfKnown["AnimalId"].isin(TestAnimals)].copy()
DfHeldOut    = DfKnown[ DfKnown["AnimalId"].isin(TestAnimals)].copy()

# Refit Wood's WITHOUT the held-out animals
WoodsParamsVal = fit_woods_params(DfKnownHeld)

# Build gap periods for held-out animals
DfGapsVal        = build_gaps(DfKnownHeld, min_gap_days=30)
AnimalsByDateVal = build_animals_by_date(DfGapsVal)

# Add held-out records as "mystery" records (strip their AnimalId)
DfHeldOut["TrueAnimalId"] = DfHeldOut["AnimalId"]
DfHeldOut["AnimalId"]     = pd.NA
DfHeldOut_idx             = DfHeldOut.index.tolist()

# Build session arrays from held-out records
SessionArraysVal = {}
for (date, session), grp in DfHeldOut.groupby(["EventDate", "MilkingSession"]):
    SessionArraysVal[(date, session)] = grp.index.to_numpy()

# Precompute predictions for held-out gap dates
WoodsPredVal = precompute_woods_predictions(WoodsParamsVal, DfGapsVal)

# Run Hungarian on held-out records
YieldStdVal = DfKnownHeld["TotalMilkYieldSessionKg"].std()
FlowStdVal  = DfKnownHeld["AverageMilkFlowKgPerMin"].std()
DurStdVal   = DfKnownHeld["MilkingDurationSeconds"].std()

ValResults = []
for Date, AbsentAnimals in AnimalsByDateVal.items():
    for Session in [1, 2, 3]:
        MysteryIdx = SessionArraysVal.get((Date, Session))
        if MysteryIdx is None or len(MysteryIdx) == 0:
            continue
        MysteryRecs = DfHeldOut.loc[MysteryIdx]
        RecYields   = MysteryRecs["TotalMilkYieldSessionKg"].values
        RecFlows    = MysteryRecs["AverageMilkFlowKgPerMin"].values
        RecDurs     = MysteryRecs["MilkingDurationSeconds"].values

        ValidAnimals = [a for a in AbsentAnimals if (a, Date) in WoodsPredVal]
        if not ValidAnimals:
            continue

        n          = max(len(ValidAnimals), len(MysteryIdx))
        CostMatrix = np.full((n, n), fill_value=1e6)
        for j, aid in enumerate(ValidAnimals):
            ExpYield   = WoodsPredVal[(aid, Date)]
            YieldDiff  = np.abs(RecYields - ExpYield)
            valid_mask = YieldDiff <= 2 * YieldStdVal
            if not valid_mask.any():
                continue
            CostMatrix[j, :len(MysteryIdx)] = (
                10 * YieldDiff / YieldStdVal +
                np.abs(RecFlows - RecFlows.mean()) / (FlowStdVal + 1e-9) +
                np.abs(RecDurs  - RecDurs.mean())  / (DurStdVal  + 1e-9)
            )
            CostMatrix[j, ~valid_mask[:len(MysteryIdx)]] = 1e6

        RowInd, ColInd = linear_sum_assignment(CostMatrix)
        for r, c in zip(RowInd, ColInd):
            if r >= len(ValidAnimals) or c >= len(MysteryIdx) or CostMatrix[r, c] >= 1e6:
                continue
            ValResults.append({
                "OriginalIndex":     MysteryIdx[c],
                "PredictedAnimalId": ValidAnimals[r],
                "TrueAnimalId":      DfHeldOut.loc[MysteryIdx[c], "TrueAnimalId"],
                "YieldError":        abs(RecYields[c] - WoodsPredVal[(ValidAnimals[r], Date)]),
            })

# Score it
DfVal       = pd.DataFrame(ValResults)
DfValConf   = DfVal[DfVal["YieldError"] < CONFIDENCE_THRESHOLD_KG]

precision   = (DfValConf["PredictedAnimalId"] == DfValConf["TrueAnimalId"]).mean() * 100
coverage    = len(DfValConf) / len(DfHeldOut) * 100

print(f" Validation Results")
print(f"Test animals          : {len(TestAnimals)}")
print(f"High-conf assignments : {len(DfValConf):,}")
print(f"Coverage              : {coverage:.1f}%   (of held-out records assigned)")
print(f"Precision             : {precision:.1f}%   (of assignments that are correct)")
print(f"Target: Precision ≥ 95%, Coverage ≥ 80%")

In [ ]:
# CELL 11 Iterative Enrichment

# Add high-confidence recovered records to the known pool and refit Wood's

RecoveredPool = (
    DfMissing.loc[DfHighConf["OriginalIndex"]]
    .assign(AnimalId=DfHighConf["RecoveredAnimalId"].values)
    .copy()
)

DfKnownEnriched = pd.concat([DfKnown, RecoveredPool], ignore_index=True)
print(f"Enriched known pool : {len(DfKnownEnriched):,} rows  "
      f"(+{len(RecoveredPool):,} recovered)")

# Refit Wood's on enriched pool
AllWoodsParamsV2 = fit_woods_params(DfKnownEnriched)

# Recompute predictions
WoodsPredDictAllV2 = precompute_woods_predictions(AllWoodsParamsV2, DfGaps)


In [ ]:
#Second Hungarian pass (V2 predictions)

ResultsV2 = []
for Date, AbsentAnimals in AnimalsByDate.items():
    for Session in [1, 2, 3]:
        MysteryIdx = SessionArrays.get((Date, Session))
        if MysteryIdx is None or len(MysteryIdx) == 0:
            continue
        MysteryRecs = DfMissing.loc[MysteryIdx]
        RecYields   = MysteryRecs["TotalMilkYieldSessionKg"].values
        RecFlows    = MysteryRecs["AverageMilkFlowKgPerMin"].values
        RecDurs     = MysteryRecs["MilkingDurationSeconds"].values

        ValidAnimals = [a for a in AbsentAnimals if (a, Date) in WoodsPredDictAllV2]
        if not ValidAnimals:
            continue

        n          = max(len(ValidAnimals), len(MysteryIdx))
        CostMatrix = np.full((n, n), fill_value=1e6)

        for j, aid in enumerate(ValidAnimals):
            ExpYield   = WoodsPredDictAllV2[(aid, Date)]
            YieldDiff  = np.abs(RecYields - ExpYield)
            valid_mask = YieldDiff <= 2 * YieldStd
            if not valid_mask.any():
                continue
            CostMatrix[j, :len(MysteryIdx)] = (
                10 * YieldDiff / YieldStd +
                np.abs(RecFlows - RecFlows.mean()) / (FlowStd + 1e-9) +
                np.abs(RecDurs  - RecDurs.mean())  / (DurStd  + 1e-9)
            )
            CostMatrix[j, ~valid_mask[:len(MysteryIdx)]] = 1e6

        RowInd, ColInd = linear_sum_assignment(CostMatrix)
        for r, c in zip(RowInd, ColInd):
            if r >= len(ValidAnimals) or c >= len(MysteryIdx) or CostMatrix[r, c] >= 1e6:
                continue
            ResultsV2.append({
                "OriginalIndex":     MysteryIdx[c],
                "RecoveredAnimalId": ValidAnimals[r],
                "Date":              Date,
                "Session":           Session,
                "AssignedCost":      CostMatrix[r, c],
                "ExpectedYield":     WoodsPredDictAllV2[(ValidAnimals[r], Date)],
                "ActualYield":       RecYields[c],
            })

DfHungarianV2 = pd.DataFrame(ResultsV2)
DfHungarianV2["YieldError"] = np.abs(DfHungarianV2["ActualYield"] - DfHungarianV2["ExpectedYield"])
DfHighConfV2  = DfHungarianV2[DfHungarianV2["YieldError"] < CONFIDENCE_THRESHOLD_KG].copy()

pct = len(DfHighConfV2) / len(DfHungarianV2) * 100 if len(DfHungarianV2) > 0 else 0
print(f"V2 total assignments     : {len(DfHungarianV2):,}")
print(f"V2 high-confidence       : {len(DfHighConfV2):,}  ({pct:.1f}%)")
print(f"Additional vs first pass : +{len(DfHighConfV2) - len(DfHighConf):,}")

In [ ]:
# CELL 12 Save + Final Export

#  Save pipeline (for inference on new data)
save_artifacts(
    ProjectRoot,
    AllWoodsParams     = AllWoodsParamsV2,       # use enriched version
    AnimalsByDate      = AnimalsByDate,
    WoodsPredDictAll   = WoodsPredDictAllV2,
    DfKnownEnriched    = DfKnownEnriched,         # known pool used for fitting
)

#  Build final output
DfFinalOutput = DfMaster.copy()
id_map = DfHighConf.set_index("OriginalIndex")["RecoveredAnimalId"].to_dict()
DfFinalOutput.loc[DfFinalOutput["AnimalId"].isna(), "AnimalId"] = (
    DfFinalOutput.loc[DfFinalOutput["AnimalId"].isna()].index.map(id_map)
)

DfFinalOutput.to_csv(ProjectRoot / "AnimalIDRecovered_Final.csv", index=False)
print("Saved → AnimalIDRecovered_Final.csv")

total_missing   = DfMaster["AnimalId"].isna().sum()
total_recovered = DfFinalOutput["AnimalId"].notna().sum() - DfKnown["AnimalId"].notna().sum()
print(f"\nRecovery summary")
print(f"  Total mystery records : {total_missing:,}")
print(f"  Recovered             : {total_recovered:,}  ({total_recovered/total_missing*100:.1f}%)")

#Part B

##Inference on New Data

In [ ]:

#  PART B — INFERENCE ON NEW DATA

#  CELL 13  Load Saved

arts = load_artifacts(
    ProjectRoot,
    "AllWoodsParams",
    "AnimalsByDate",
    "WoodsPredDictAll",
    "DfKnownEnriched",
)
AllWoodsParams   = arts["AllWoodsParams"]
AnimalsByDate    = arts["AnimalsByDate"]
WoodsPredDictAll = arts["WoodsPredDictAll"]
DfKnown          = arts["DfKnownEnriched"]

In [ ]:

# CELL 14 Load & Clean NEW Data

NEW_DataFilePath = ProjectRoot / "YOUR_NEW_FARM_FILE.csv"   # ← CHANGE THIS

DfKnownNew, DfMissingNew, DfMasterNew = load_and_clean(NEW_DataFilePath)

In [ ]:
# CELL 15 Detect Gaps in New Dat

DfGapsNew         = build_gaps(DfKnownNew, min_gap_days=30)
AnimalsByDateNew  = build_animals_by_date(DfGapsNew)
SessionArraysNew  = build_session_arrays(DfMissingNew)

# Optional: compute fresh Wood's predictions for new date range
WoodsPredNew = precompute_woods_predictions(AllWoodsParams, DfGapsNew)

In [ ]:

# CELL 16-17 Hungarian + Confidence Filter on New Data


In [ ]:
# Then export:
#   DfFinalOutputNew.to_csv(ProjectRoot / "AnimalIDRecovered_NewData.csv")